# Cuaderno 10. Comparación de OCR: impresos, mecanografiados y manuscritos

## 0. Instalar dependencias (solo Colab)

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run("apt-get -qq install -y tesseract-ocr > /dev/null", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.5.4", "accelerate", "bitsandbytes",
                    "python-doctr", "pytesseract", "jiwer"], check=True)
    print("Instalados Tesseract, transformers, docTR y jiwer")
else:
    print("Fuera de Colab no se instala nada: ver los paquetes en cuadernos/README.md.")

## 1. Funciones de texto

`limpiar()` quita las marcas de formato (XML, HTML, Markdown), une las palabras cortadas con guion al final de línea y reduce los espacios y saltos de línea a uno. `normalizar()` además pasa a minúsculas, quita tildes y puntuación, y lleva las letras antiguas (`ſ`, `u`/`v`, `i`/`j`/`y`) a sus formas modernas; no desarrolla abreviaturas.

In [ ]:
import html
import re


def limpiar(texto):
    """Deja solo el texto: sin marcas XML, HTML o Markdown, con las palabras cortadas
    por guion al final de línea unidas, y con un solo espacio entre palabras."""
    texto = re.sub(r"<Metadata>.*?</Metadata>", "", texto, flags=re.S)  # ficha que Churro agrega antes del texto
    texto = re.sub(r"</(Line|p|div|li|h\d)>|<br\s*/?>|<Gap[^>]*/>", " ", texto)  # fin de línea o de párrafo
    texto = re.sub(r"<[^>]+>", "", texto)
    texto = html.unescape(texto).replace("**", "")
    texto = re.sub(r"^#+\s*", "", texto, flags=re.M)
    texto = re.sub(r"(\w)-\s+(?=[a-záéíóúüñ])", r"\1", texto)  # "pala- bra" -> "palabra"
    return re.sub(r"\s+", " ", texto).strip()

import unicodedata

LETRAS = str.maketrans({"ſ": "s", "ꝛ": "r", "ꝺ": "d", "ꝑ": "p", "ꝓ": "p", "ꝗ": "q", "ꝙ": "q", "⁊": "e",
                        "&": "e", "v": "u", "j": "i", "y": "i", "ß": "ss", "æ": "ae", "œ": "oe"})


def normalizar(texto):
    """Normalización gráfica para escrituras antiguas: minúsculas, sin tildes ni signos de
    abreviatura, sin puntuación, letras antiguas llevadas a las modernas y u/v, i/j/y unificadas.
    Las abreviaturas NO se desarrollan: "q̃" sigue siendo distinto de "que"."""
    texto = re.sub(r"(\w)[-=¬]\s+(?=\w)", r"\1", texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^\w\s]|_", " ", texto.translate(LETRAS))
    return re.sub(r"\s+", " ", texto).strip()

## 2. Elegir y cargar los documentos

`TIPOS` define qué tipos de documento se procesan (cinco de cada uno).

In [ ]:
TIPOS = ["imprenta", "mecanografiado", "manuscritos"]

Carga los documentos de los tipos elegidos. Imprenta y manuscritos se descargan de [CHURRO-DS](https://huggingface.co/datasets/stanford-oval/churro-dataset) (Stanford; licencia de solo uso de investigación); la referencia es el texto de cada `<Line>` de la transcripción. Los mecanografiados son párrafos del [OCR Berrutti Dataset v3](https://github.com/camilomarino/ocr_berrutti_dataset) (Universidad de la República / Centre Borelli, ENS Paris-Saclay; licencia CC BY-NC-SA 4.0), guardados en `datos_ocr/mecanografiado/` con su transcripción.

In [ ]:
import io
import urllib.request
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image

# Páginas de CHURRO-DS: archivo del dataset que las contiene y su identificador.
CHURRO = {
    "imprenta": ("data/test-00001-of-00003.parquet", [
        "fondue-chapbooks/Varios_435_16",  # pliego de cordel, siglo XIX
        "fondue-es-print-19/antonio_jose_valdes_delirio_ii_Delirio II, en La prensa argentina 18 jun 1816_page-0008",  # Antonio José Valdés, «Delirio II», La Prensa Argentina, 1816
        "fondue-chapbooks/Varios_435_6",  # pliego de cordel, siglo XIX
        "fondue-es-print-19/ricardo_palma_apocaliptica_1883_Ricardo Palma - Apocaliptica",  # Ricardo Palma, «Apocalíptica», 1883
        "fondue-es-print-19/juan_egana_progresos_de_la_civilizacion_ocios_filosoficos_y_poeticos_ixe4bcu_pdf_page_85",  # Juan Egaña, Ocios filosóficos y poéticos, p. 85
    ]),
    "manuscritos": ("data/test-00000-of-00003.parquet", [
        "codea/CODEA-3577",  # documento notarial (corpus CODEA)
        "spain/bnf-esp-256_fuero_juzgo_6v",  # Fuero Juzgo, BnF Espagnol 256, f. 6v (códice medieval)
        "notarial/170025120000003,0052",  # protocolo notarial
        "codea/CODEA-2561",  # documento notarial (corpus CODEA)
        "aldicam/ALDICAM-0341",  # documento de archivo (corpus ALDICAM)
    ]),
}

# Párrafos de Berrutti guardados en el repositorio del taller.
MECANOGRAFIADO = [
    "high_r0809_1437_p01",
    "high_r0809_1437_p04",
    "high_r0126_0309_p02",
    "high_r0813_0921_p01",
    "high_r0814_0185_p01",
]
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/datos_ocr/mecanografiado/"
CARPETA = Path("datos_ocr/mecanografiado")  # fuera de Colab se leen del repositorio clonado


def leer(nombre):
    if not (CARPETA / nombre).exists():
        CARPETA.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + nombre, CARPETA / nombre)
    return CARPETA / nombre


paginas = []
for tipo in TIPOS:
    if tipo == "mecanografiado":
        for id_ in MECANOGRAFIADO:
            imagen = Image.open(leer(id_ + ".png")).convert("RGB")
            referencia = leer(id_ + ".txt").read_text(encoding="utf-8")
            paginas.append({"tipo": tipo, "id": id_, "imagen": imagen, "referencia": limpiar(referencia)})
        continue
    archivo, ids = CHURRO[tipo]
    parquet = hf_hub_download("stanford-oval/churro-dataset", archivo, repo_type="dataset")
    filas = pd.read_parquet(parquet, columns=["example_id", "image", "cleaned_transcription"],
                            filters=[("example_id", "in", ids)]).set_index("example_id").loc[ids]
    for id_, fila in filas.iterrows():
        imagen = Image.open(io.BytesIO(fila["image"]["bytes"])).convert("RGB")
        imagen.thumbnail((2048, 2048))  # el lado mayor queda en 2048 píxeles, como en la evaluación
        # La referencia es el texto de cada <Line> de la transcripción XML.
        lineas = re.findall(r"<Line\b[^>]*>(.*?)</Line>", fila["cleaned_transcription"], re.S)
        paginas.append({"tipo": tipo, "id": id_, "imagen": imagen, "referencia": limpiar(" ".join(lineas))})

print(pd.Series([p["tipo"] for p in paginas]).value_counts().to_string())

In [ ]:
from IPython.display import display

for pagina in paginas:
    miniatura = pagina["imagen"].copy()
    miniatura.thumbnail((600, 600))
    print(f"[{pagina['tipo']}] {pagina['id']} - {len(pagina['referencia'])} caracteres en la referencia")
    display(miniatura)

## 3. OCR clásicos: docTR y Tesseract

Carga [docTR](https://github.com/mindee/doctr) y [Tesseract](https://github.com/tesseract-ocr/tesseract) y define una función de transcripción para cada uno. Tesseract usa el modelo de español `tessdata_best` y `--psm 6` (la imagen como un solo bloque de texto).

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np
import pytesseract
import torch
from doctr.models import ocr_predictor

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

# docTR: detección db_resnet50 + reconocimiento crnn_vgg16_bn
modelo_doctr = ocr_predictor(pretrained=True).to(DISPOSITIVO)


def transcribir_doctr(imagen):
    pagina = modelo_doctr([np.array(imagen)]).export()["pages"][0]
    lineas = [" ".join(palabra["value"] for palabra in linea["words"])
              for bloque in pagina["blocks"] for linea in bloque["lines"]]
    return "\n".join(lineas)


# Tesseract con el modelo de español más preciso (tessdata_best) y --psm 6: un bloque de texto uniforme.
TESSDATA = Path("tessdata")
if not (TESSDATA / "spa.traineddata").exists():
    TESSDATA.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://github.com/tesseract-ocr/tessdata_best/raw/main/spa.traineddata",
                               TESSDATA / "spa.traineddata")


def transcribir_tesseract(imagen):
    return pytesseract.image_to_string(imagen, lang="spa", config=f"--tessdata-dir {TESSDATA} --psm 6")


print("Tesseract", pytesseract.get_tesseract_version(), "- docTR en", DISPOSITIVO)

## 4. Modelos de visión y lenguaje

`cargar()` descarga un modelo y lo pone en la GPU; con `cuantizar=True` lo carga en 4 bits. `transcribir()` arma el mensaje (imagen, instrucción de sistema e instrucción de usuario, según el modelo), genera el texto y lo devuelve crudo.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

# bfloat16 en GPU modernas (L4, A100); float16 en la T4, que no tiene bfloat16 nativo; float32 sin GPU.
if not torch.cuda.is_available():
    TIPO = torch.float32
elif torch.cuda.get_device_capability()[0] >= 8:
    TIPO = torch.bfloat16
else:
    TIPO = torch.float16


def cargar(repo, cuantizar=False, codigo_propio=False, **_):
    opciones = {"dtype": TIPO, "device_map": DISPOSITIVO, "trust_remote_code": codigo_propio}
    if cuantizar:  # 4 bits (NF4): el modelo ocupa la cuarta parte de memoria
        opciones["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                             bnb_4bit_compute_dtype=TIPO)
    clase = AutoModelForCausalLM if codigo_propio else AutoModelForImageTextToText
    return clase.from_pretrained(repo, **opciones), AutoProcessor.from_pretrained(repo, trust_remote_code=codigo_propio)


def transcribir(modelo, procesador, pagina, sistema=None, instruccion=None, muestreo=None, plantilla=None, **_):
    if isinstance(instruccion, dict):  # una instrucción distinta para cada tipo de documento
        instruccion = instruccion[pagina["tipo"]]
    contenido = [{"type": "image", "image": pagina["imagen"]}]
    if instruccion:
        contenido.append({"type": "text", "text": instruccion})
    mensajes = [{"role": "user", "content": contenido}]
    if sistema:
        mensajes.insert(0, {"role": "system", "content": [{"type": "text", "text": sistema}]})
    entradas = procesador.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=True, return_dict=True,
                                              return_tensors="pt", **(plantilla or {})).to(modelo.device)
    # Todos estos modelos derivan de Qwen y terminan cada respuesta con <|im_end|>.
    fin = procesador.tokenizer.convert_tokens_to_ids("<|im_end|>")
    salida = modelo.generate(**entradas, max_new_tokens=MAX_TOKENS, eos_token_id=fin, **(muestreo or {"do_sample": False}))
    return procesador.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

`MODELOS` define qué modelos se corren, con la instrucción y el muestreo de cada uno. Borrá una entrada para no correr ese modelo. Qwen3-VL-8B se carga en 4 bits para que entre en una T4. `MAX_TOKENS` limita el texto generado por página.

In [ ]:
MAX_TOKENS = 4000  # tope de texto generado por página; corta los bucles de repetición

# Instrucciones de nuestra evaluación para el modelo de propósito general, una por tipo de documento.
PROMPT_IMPRESO = (
    "OCR this scanned column from a historical printed Spanish-language newspaper. Return only "
    "plain text. Preserve original line breaks, spelling (including printing errors), accents, "
    "capitalization, punctuation, abbreviations, dates, numbers, and hyphenation. Do not "
    "explain. Do not summarize. Do not use markdown. Do not correct or expand abbreviations. Do "
    "not invent missing text. Use [ilegible] only for unreadable text."
)
PROMPT_MECANOGRAFIADO = (
    "OCR this scanned paragraph from a historical typewritten Uruguayan document. Return only "
    "plain text. Preserve original line breaks, spelling, accents, capitalization, punctuation, "
    "abbreviations, dates, numbers, and hyphenation. Do not explain. Do not summarize. Do not "
    "use markdown. Do not correct or expand abbreviations. Do not invent missing text. Use "
    "[ilegible] only for unreadable text."
)
PROMPT_MANUSCRITO = (
    "OCR this scanned page from a historical handwritten Spanish document. Return only plain "
    "text. Preserve original line breaks, spelling, capitalization, punctuation, dates and "
    "numbers. Keep abbreviations exactly as written, including abbreviation marks and special "
    "letters (for example q̃, ꝑ, ⁊, ſ); do not expand them. Do not modernize or correct the "
    "text. Do not explain. Do not summarize. Do not use markdown. Do not invent missing text. "
    "Use [ilegible] only for unreadable text."
)

# Final de la instrucción oficial de Chandra (repositorio datalab-to/chandra).
CHANDRA_FIN = """Only use these tags ['math', 'br', 'i', 'b', 'u', 'del', 'sup', 'sub', 'table', 'tr', 'td', 'p', 'th', 'div', 'pre', 'h1', 'h2', 'h3', 'h4', 'h5', 'ul', 'ol', 'li', 'input', 'a', 'span', 'img', 'hr', 'tbody', 'small', 'caption', 'strong', 'thead', 'big', 'code', 'chem'], and these attributes ['class', 'colspan', 'rowspan', 'display', 'checked', 'type', 'border', 'value', 'style', 'href', 'alt', 'align', 'data-bbox', 'data-label'].

Guidelines:
* Inline math: Surround math with <math>...</math> tags. Math expressions should be rendered in KaTeX-compatible LaTeX. Use display for block math.
* Tables: Use colspan and rowspan attributes to match table structure.
* Formatting: Maintain consistent formatting with the image, including spacing, indentation, subscripts/superscripts, and special characters.
* Images: Include a description of any images in the alt attribute of an <img> tag. Do not fill out the src property. Describe in detail inside the div tag. Also convert charts to high fidelity data, and convert diagrams to mermaid.
* Forms: Mark checkboxes and radio buttons properly.
* Text: join lines together properly into paragraphs using <p>...</p> tags.  Use <br> tags for line breaks within paragraphs, but only when absolutely necessary to maintain meaning.
* Chemistry: Use <chem>...</chem> tags for chemical formulas with reactive SMILES.
* Lists: Preserve indents and proper list markers.
* Use the simplest possible HTML structure that accurately represents the content of the block.
* Make sure the text is accurate and easy for a human to read and interpret.  Reading order should be correct and natural."""

# Borrá una entrada para no correr ese modelo. Se cargan de a uno: en una T4 no entran dos juntos.
MODELOS = {
    "Churro-3B": {"repo": "stanford-oval/churro-3B",
                  "sistema": "Transcribe the entirety of this historical document to XML format."},
    "LightOnOCR-3-4B": {"repo": "lightonai/LightOnOCR-3-4B",  # sin instrucción: solo transcribe
                        "plantilla": {"enable_thinking": False},
                        "muestreo": {"do_sample": True, "temperature": 0.2, "top_p": 0.9}},
    "Qwen3-VL-8B (4 bits)": {"repo": "Qwen/Qwen3-VL-8B-Instruct", "cuantizar": True,
                             "instruccion": {"imprenta": PROMPT_IMPRESO, "mecanografiado": PROMPT_MECANOGRAFIADO,
                                             "manuscritos": PROMPT_MANUSCRITO}},
    "Chandra OCR 2": {"repo": "datalab-to/chandra-ocr-2", "instruccion": "OCR this image to HTML.\n\n" + CHANDRA_FIN,
                      "plantilla": {"enable_thinking": False}},
}

## 5. Transcribir

Define dónde se guardan las transcripciones: en `resultados/10_ocr_comparacion/` o, con `GUARDAR_EN_DRIVE = True`, en Google Drive (Colab pide permiso para acceder).

In [ ]:
GUARDAR_EN_DRIVE = False  # True: guarda en Google Drive, para retomar aunque Colab se desconecte

if EN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SALIDA = Path("/content/drive/MyDrive/sicss_ocr/10_ocr_comparacion")
else:
    SALIDA = Path("resultados") / "10_ocr_comparacion"
TEXTOS = SALIDA / "textos"
TEXTOS.mkdir(parents=True, exist_ok=True)
TIEMPOS = SALIDA / "tiempos.tsv"
print("Las transcripciones se guardan en", SALIDA)


def archivo_de(pagina, sistema):
    return TEXTOS / (re.sub(r"[^\w.-]+", "_", pagina["id"] + "__" + sistema) + ".txt")


def pendientes(sistema):
    return [p for p in paginas if not archivo_de(p, sistema).exists()]

Pasa cada documento por los seis sistemas y guarda cada transcripción apenas termina. Al volver a ejecutar, se saltean las que ya están. Los modelos de visión y lenguaje se cargan de a uno y se liberan antes del siguiente.

In [ ]:
def medir(sistema, transcribir_pagina):
    for pagina in pendientes(sistema):
        inicio = time.perf_counter()
        texto = transcribir_pagina(pagina)
        segundos = round(time.perf_counter() - inicio, 1)
        archivo_de(pagina, sistema).write_text(texto, encoding="utf-8")  # se guarda apenas termina
        with TIEMPOS.open("a", encoding="utf-8") as tiempos:
            tiempos.write(f"{pagina['id']}\t{sistema}\t{segundos}\n")
        print(f"{sistema:>22}  {segundos:6.1f} s  [{pagina['tipo']}] {pagina['id']}")


medir("docTR", lambda pagina: transcribir_doctr(pagina["imagen"]))
medir("Tesseract", lambda pagina: transcribir_tesseract(pagina["imagen"]))
for nombre, opciones in MODELOS.items():
    if not pendientes(nombre):  # ya transcribió todas las páginas en una ejecución anterior
        continue
    modelo, procesador = cargar(**opciones)
    torch.manual_seed(0)
    medir(nombre, lambda pagina: transcribir(modelo, procesador, pagina, **opciones))
    del modelo, procesador  # libera la GPU antes de cargar el modelo siguiente
    gc.collect()
    torch.cuda.empty_cache()

## 6. CER y WER

Calcula el CER, el WER y el CER normalizado de cada sistema en cada documento contra la referencia, después de aplicar `limpiar()` a las dos (y `normalizar()`, para el normalizado). **CER**: ediciones de un carácter (sustituir, borrar o agregar) sobre la cantidad de caracteres de la referencia; **WER**: lo mismo en palabras. 0 % es una transcripción perfecta. Muestra el CER mediano por tipo de documento y, para cada tipo, el resumen y el detalle por documento. El **CER total** suma todos los errores sobre todos los caracteres.

In [ ]:
import jiwer

SISTEMAS = ["docTR", "Tesseract", *MODELOS]
tiempos = pd.read_csv(TIEMPOS, sep="\t", names=["pagina", "sistema", "segundos"]).drop_duplicates(["pagina", "sistema"], keep="last")
transcripciones = pd.DataFrame([{"tipo": p["tipo"], "pagina": p["id"], "sistema": s, "texto": archivo_de(p, s).read_text(encoding="utf-8")}
                                for p in paginas for s in SISTEMAS])
transcripciones = transcripciones.merge(tiempos, on=["pagina", "sistema"], how="left")

referencias = {p["id"]: p["referencia"] for p in paginas}
transcripciones["limpio"] = transcripciones["texto"].map(limpiar)
pares = list(zip(transcripciones["pagina"], transcripciones["limpio"]))
transcripciones["CER"] = [100 * jiwer.cer(referencias[p], t) for p, t in pares]
transcripciones["WER"] = [100 * jiwer.wer(referencias[p], t) for p, t in pares]
transcripciones["CER normalizado"] = [100 * jiwer.cer(normalizar(referencias[p]), normalizar(t)) for p, t in pares]

print("CER mediano (%) por tipo de documento:")
display(transcripciones.pivot_table(index="sistema", columns="tipo", values="CER", aggfunc="median").loc[SISTEMAS, TIPOS].round(2))

for tipo, grupo in transcripciones.groupby("tipo", sort=False):
    resumen = grupo.groupby("sistema")[["CER", "WER", "CER normalizado", "segundos"]].median().round(2).add_suffix(" (mediana)")
    for sistema, filas in grupo.groupby("sistema"):
        resumen.loc[sistema, "CER total"] = round(100 * jiwer.cer([referencias[p] for p in filas["pagina"]], list(filas["limpio"])), 2)
    print(f"\n{tipo.upper()} (%; segundos por página):")
    display(resumen.sort_values("CER total"))
    print("CER por página (%):")
    display(grupo.pivot(index="pagina", columns="sistema", values="CER")[SISTEMAS].round(2))

## 7. Comparar un documento

Muestra la imagen, la referencia y las transcripciones limpias. Cambiá `PAGINA` para ver otro: primero van los impresos, después los mecanografiados y al final los manuscritos.

In [ ]:
PAGINA = 0  # número de página, de 0 a len(paginas) - 1

pagina = paginas[PAGINA]
print(f"[{pagina['tipo']}] {pagina['id']}")
display(pagina["imagen"])
print("REFERENCIA\n" + pagina["referencia"] + "\n")
for _, fila in transcripciones[transcripciones["pagina"] == pagina["id"]].iterrows():
    print(f"{fila['sistema'].upper()} (CER {fila['CER']:.2f} %)\n{fila['limpio']}\n")

## 8. Descargar los resultados

Guarda las métricas por documento en `metricas_por_pagina.csv`, junto a los textos crudos (`textos/`) y los tiempos (`tiempos.tsv`). En Colab descarga todo como `.zip`.

In [ ]:
import shutil

transcripciones.drop(columns="texto").to_csv(SALIDA / "metricas_por_pagina.csv", index=False)
print("Resultados en", SALIDA)

if EN_COLAB:
    from google.colab import files
    files.download(shutil.make_archive("10_ocr_comparacion", "zip", SALIDA))